In [1]:
# Imports (todos nesta célula)
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score)

# A preparação escolhida nas tarefas 2 e 3 e a régua de avaliação estão em preprocessing.py,
# o mesmo ficheiro que o 02_preprocessing importa (como o preprocessing.py das práticas)
from preprocessing import (ENCODER_FINAL, SCALER_FINAL, TARGET, compare, evaluate, fit_predict,
                           load_train, prepare_selected, sem, time_splits, verdict)

In [2]:
# ===== [4.0] FERRAMENTAS - início =====
# Passo 1 - os dados e as 5 divisões por tempo, exatamente como no 02 (passos 1 a 3): o treino
# ordenado por data, sem as 65 linhas sem data, e cada divisão treina no passado e valida no
# que vem a seguir (W5, secção 3.4)
raw, frame = load_train("../data/train.csv")
splits = time_splits(frame)
test_csv = pd.read_csv("../data/test.csv", index_col="ID")

print(f"treino: {len(frame):,} compras ({len(raw) - len(frame)} sem data ficam de fora);"
      f" teste: {len(test_csv):,} compras")
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    print(f"divisão {fold}: treina com {len(train_idx):,}, valida com {len(val_idx):,}")

treino: 6,469 compras (65 sem data ficam de fora); teste: 986 compras
divisão 1: treina com 1,079, valida com 1,078
divisão 2: treina com 2,157, valida com 1,078
divisão 3: treina com 3,235, valida com 1,078
divisão 4: treina com 4,313, valida com 1,078
divisão 5: treina com 5,391, valida com 1,078


In [3]:
# Passo 2 - verificação: a preparação importada tem de dar exatamente o resultado do 02
# (passo 62: F1 0.7508, accuracy 0.7163). Se o preprocessing.py mudar sem querer, o notebook
# pára aqui com um erro.
selected_run = evaluate(prepare_selected, frame, splits, scaler=SCALER_FINAL,
                        encoder=ENCODER_FINAL)
assert round(selected_run["f1"].mean(), 4) == 0.7508
assert round(selected_run["accuracy"].mean(), 4) == 0.7163
print(f"referência (regressão logística, preparação final): F1 {selected_run['f1'].mean():.4f}"
      f" +- {sem(selected_run['f1']):.4f}, accuracy {selected_run['accuracy'].mean():.4f}")

referência (regressão logística, preparação final): F1 0.7508 +- 0.0074, accuracy 0.7163


In [4]:
# Passo 3 - a régua dos modelos. Nas aulas, cada definição de um modelo é medida no treino e na
# validação, lado a lado (W6 a W9, "The settings of ..."): um F1 de treino muito acima do de
# validação é sobreajuste; dois valores baixos e próximos são subajuste (W5, "The Complete
# Model Selection Workflow", ponto 2). A tarefa 3 mostrou que também é preciso ver quantas
# vezes o modelo prevê 1 (passos 47 e 59 do 02).
def run_model(model, threshold=None):
    """Em cada divisão: F1 e accuracy na validação, F1 no treino e % de previsões 1.

    `model=None` é a regressão logística da referência; `threshold=None` usa o limiar 0.5.
    """
    rows = []
    for train_idx, val_idx in splits:
        train, valid = frame.iloc[train_idx], frame.iloc[val_idx]
        predicted = fit_predict(prepare_selected, train, valid, SCALER_FINAL, ENCODER_FINAL,
                                model, None, threshold)
        on_train = fit_predict(prepare_selected, train, train, SCALER_FINAL, ENCODER_FINAL,
                               model, None, threshold)
        rows.append({"f1": f1_score(valid[TARGET], predicted),
                     "accuracy": accuracy_score(valid[TARGET], predicted),
                     "treino f1": f1_score(train[TARGET], on_train),
                     "% de 1": 100 * predicted.mean()})
    return pd.DataFrame(rows)


def sweep(models, threshold=None):
    """Cada modelo contra a referência, nas mesmas 5 divisões, com a regra dos 2 x SEM."""
    runs = {name: run_model(model, threshold) for name, model in models.items()}
    table = compare(runs, reference, metric="f1").astype({"f1": float, "accuracy": float,
                                                           "vs base": float, "SEM": float})
    table["2 x SEM"] = 2 * table["SEM"]
    table["veredicto"] = verdict(table)
    table["F1 treino"] = [runs[name]["treino f1"].mean() for name in table.index]
    table["treino - validação"] = table["F1 treino"] - table["f1"]
    table["% de 1"] = [runs[name]["% de 1"].mean() for name in table.index]
    shown = ["f1", "vs base", "2 x SEM", "melhor em", "veredicto", "F1 treino",
             "treino - validação", "accuracy", "% de 1"]
    return table[shown], runs


reference = run_model(None)
assert (reference["f1"] == selected_run["f1"]).all()   # a mesma régua que o evaluate
check, _ = sweep({"regressão logística (referência)": None})
print(check.round(4).to_string())

                                      f1  vs base  2 x SEM melhor em veredicto  F1 treino  treino - validação  accuracy  % de 1
regressão logística (referência)  0.7508      0.0      0.0    0 de 5    empata     0.7316             -0.0192    0.7163  57.885


In [5]:
# ===== [4.0] FERRAMENTAS - fim =====
# Passo 4 - o que o 4.0 entrega aos passos seguintes:
#   - reference: a regressão logística por defeito, F1 0.7508, accuracy 0.7163
#   - sweep({"nome": modelo, ...}): a tabela de cada pacote (4.2 a 4.5) contra a referência
#   - threshold=: o limiar de decisão, para o 4.6 (W5, secção 2.7)
# A referência tem F1 de treino 0.7316, abaixo do de validação (0.7508): não há sinal de
# sobreajuste. Um modelo mais flexível pode aproveitar mais dos dados; é o que o 4.2 a 4.5 testam.
print("4.0 pronto: reference, run_model e sweep")

4.0 pronto: reference, run_model e sweep


In [6]:
# ===== [4.1] ESTRATÉGIA E MÉTRICA - início =====
# Passo 5 - a estratégia de avaliação: 5 divisões por tempo (TimeSeriesSplit, W5, secção 3.4).
# O homework pede uma estratégia e que se explique porque serve este problema:
#   1. o test.csv é posterior ao train.csv (02, passo 2): a pergunta real é "prever o futuro a
#      partir do passado", e cada divisão faz exatamente isso
#   2. 94.5% das compras do teste são de clientes que já aparecem no treino (02, passo 2): o
#      modelo vai prever clientes que conhece, mas sempre em datas posteriores
#   3. o holdout e o K-Fold ao acaso (W5, secções 2 e 3.1) poriam compras do futuro no treino e
#      avaliariam com o passado, o que dá resultados melhores do que os reais
#   - 5 divisões em vez de 1: um só corte é "uma amostra de tamanho um" (W5, key takeaways);
#     com 5 vê-se a média e o erro-padrão, e as diferenças comparam-se divisão a divisão
#   - limitação conhecida: não há intervalo entre treino e validação; o alvo de uma compra de
#     treino pode olhar 90 dias para dentro da validação (02, passo 3)
#   - o test.csv nunca é usado para escolher nada: só para a previsão final (W5, secção 2.1)
print("estratégia: TimeSeriesSplit(n_splits=5), treino sempre anterior à validação")

estratégia: TimeSeriesSplit(n_splits=5), treino sempre anterior à validação


In [7]:
# Passo 6 - as métricas da W5 (secções 2.2 a 2.5) para a referência e para o piso "prever
# sempre 1" (W5, célula 110), divisão a divisão. Precision: das compras que o modelo diz que
# vão ter recompra, quantas têm. Recall: das compras com recompra, quantas o modelo encontra.
metric_rows = []
for fold, (train_idx, val_idx) in enumerate(splits, start=1):
    train, valid = frame.iloc[train_idx], frame.iloc[val_idx]
    truth = valid[TARGET]
    for name, predicted in [("referência", fit_predict(prepare_selected, train, valid,
                                                       SCALER_FINAL, ENCODER_FINAL)),
                            ("prever sempre 1", np.ones(len(valid), dtype=int))]:
        metric_rows.append({"divisão": fold, "previsão": name,
                            "precision": precision_score(truth, predicted),
                            "recall": recall_score(truth, predicted),
                            "f1": f1_score(truth, predicted),
                            "accuracy": accuracy_score(truth, predicted),
                            "% de 1 real": 100 * truth.mean()})
metrics = pd.DataFrame(metric_rows)
shown = ["precision", "recall", "f1", "accuracy"]
print("referência, divisão a divisão:")
print(metrics[metrics["previsão"] == "referência"].set_index("divisão")[shown + ["% de 1 real"]]
      .round(4).to_string(), "\n")
print("média das 5 divisões:")
print(metrics.groupby("previsão")[shown].mean().round(4).to_string())

referência, divisão a divisão:
         precision  recall      f1  accuracy  % de 1 real
divisão                                                  
1           0.7008  0.8365  0.7627    0.6957      58.4416
2           0.7178  0.7639  0.7401    0.7134      53.4323
3           0.7791  0.7327  0.7552    0.7263      57.6067
4           0.7359  0.7201  0.7279    0.7199      52.0408
5           0.7862  0.7512  0.7683    0.7263      60.3896 

média das 5 divisões:
                 precision  recall      f1  accuracy
previsão                                            
prever sempre 1     0.5638  1.0000  0.7206    0.5638
referência          0.7439  0.7609  0.7508    0.7163


In [8]:
# Passo 7 - a matriz de confusão da referência (W5, secção 2.1, Step 26), somando as 5
# validações. O scikit-learn arruma-a assim: [[TN FP] [FN TP]]
#   TN: sem recompra, previsto sem recompra   FP: sem recompra, previsto com (falso alarme)
#   FN: com recompra, previsto sem (falha)    TP: com recompra, previsto com recompra
matrix = sum(confusion_matrix(frame[TARGET].iloc[val_idx],
                              fit_predict(prepare_selected, frame.iloc[train_idx],
                                          frame.iloc[val_idx], SCALER_FINAL, ENCODER_FINAL))
             for train_idx, val_idx in splits)
matrix_table = pd.DataFrame(matrix, index=["real 0 (sem recompra)", "real 1 (com recompra)"],
                            columns=["previsto 0", "previsto 1"])
print(matrix_table.to_string(), "\n")
(tn, fp), (fn, tp) = matrix
print(f"{tn + fp + fn + tp:,} compras de validação; acertos {tn + tp:,}"
      f" ({100 * (tn + tp) / matrix.sum():.1f}%)")
print(f"falsos alarmes (FP): {fp:,} — {100 * fp / (tn + fp):.1f}% das compras sem recompra")
print(f"falhas (FN): {fn:,} — {100 * fn / (fn + tp):.1f}% das compras com recompra")

                       previsto 0  previsto 1
real 0 (sem recompra)        1546         805
real 1 (com recompra)         724        2315 

5,390 compras de validação; acertos 3,861 (71.6%)
falsos alarmes (FP): 805 — 34.2% das compras sem recompra
falhas (FN): 724 — 23.8% das compras com recompra


In [9]:
# ===== [4.1] ESTRATÉGIA E MÉTRICA - fim =====
# Passo 8 - a métrica: F1 (escolha da equipa), com a accuracy sempre ao lado.
#   - o F1 é a média harmónica da precision e da recall da classe 1, a compra com recompra
#     (W5, secção 2.5): só é alto se o modelo encontrar quem volta (recall) sem encher a lista
#     de falsos alarmes (precision). É a classe que interessa à empresa: o enunciado do projeto
#     quer prever a recompra para "more targeted commercial actions"
#   - a referência equilibra as duas: precision 0.744, recall 0.761; os erros dividem-se entre
#     805 falsos alarmes e 724 falhas (passo 7)
#   - o limite do F1, que a própria W5 aponta (célula 111): quando a classe positiva é a
#     maioria, prever sempre 1 já tem F1 alto. Aqui 56% das compras têm recompra, e prever
#     sempre 1 dá F1 0.721; a referência só lhe ganha +0.030 no F1, mas +0.153 na accuracy
#     (0.564 -> 0.716). Por isso a accuracy aparece sempre ao lado: mostra que o modelo
#     aprendeu, coisa que o F1 sozinho quase esconde
#   - a accuracy também serve aqui (as classes estão perto de 56/44, não muito desequilibradas),
#     mas trata da mesma forma acertar quem volta e quem não volta
#   - a confirmar: a métrica da competição no Kaggle ("Overview" -> "Evaluation")
print("métrica: F1 (decide); accuracy, precision e recall ao lado")

métrica: F1 (decide); accuracy, precision e recall ao lado


In [10]:
# ===== SUBMISSÃO AO KAGGLE - início =====
# Passo 9 - a célula que gera o ficheiro passou do 02 para aqui: o modelo escolhe-se neste
# notebook, e o enunciado do projeto (V. Deployment) pede que a última parte aplique o pipeline
# ao test.csv. Primeiro, a prova de que nada mudou na passagem: sem limiar, a preparação final
# e a regressão logística, ajustadas no treino inteiro, dão exatamente a Version04.
version_04 = pd.read_csv("../submissions/ML_Group_37_Version04.csv", index_col="ID")[TARGET]
same_as_04 = fit_predict(prepare_selected, frame, test_csv, SCALER_FINAL, ENCODER_FINAL)
assert (same_as_04 == version_04.loc[test_csv.index].to_numpy()).all()
print("sem limiar, o 03 reproduz a Version04: 986 de 986 previsões iguais")

sem limiar, o 03 reproduz a Version04: 986 de 986 previsões iguais


In [11]:
# Passo 10 - a submissão. Histórico (score = a parte visível da leaderboard, 70% do teste):
#   Version01  preparação do 2.2, regressão logística                 F1 local 0.734  Kaggle 0.77
#   Version02  as mesmas previsões da Version01                                        Kaggle ?
#   Version03  preparação final do 2.7                                F1 local 0.748  Kaggle ?
#   Version04  + as features da tarefa 3 (sem as 14 colunas)          F1 local 0.751  Kaggle ?
#   Version05  a Version04 com limiar 0.45 (W5, secção 2.7)            F1 local 0.753  Kaggle ?
# A Version05 é uma sonda, não uma melhoria provada: localmente o limiar 0.45 empata no F1
# (+0.0026, 2 x SEM = 0.0043) e piora a accuracy (-0.011). Prevê mais 1s (697 contra 649),
# como o sample_submission (741), que teve 0.779 contra os 0.77 da Version01. Se subir, é sinal
# de que a métrica do Kaggle premeia prever mais 1s. Não se escolhe o modelo final por isto.
# VERSION e THRESHOLD são escritos à mão: correr o notebook outra vez volta a gerar o mesmo
# ficheiro, e o notebook entregue tem de reproduzir a submissão escolhida no Kaggle.
TEAM = "ML_Group_37"
VERSION = 5
THRESHOLD = 0.45   # None = o predict do modelo (limiar 0.5)

predictions = fit_predict(prepare_selected, frame, test_csv, SCALER_FINAL, ENCODER_FINAL,
                          threshold=THRESHOLD)
submission = pd.DataFrame({"ID": test_csv.index, TARGET: predictions})

path = f"../submissions/{TEAM}_Version{VERSION:02d}.csv"
submission.to_csv(path, index=False)   # sem o índice do pandas, como pede o guia do Kaggle

sample = pd.read_csv("../data/sample_submission.csv")
print(f"gravado em {path}")
print(f"linhas: {len(submission)}, IDs únicos: {submission['ID'].nunique()},"
      f" mesmos IDs do sample_submission: {submission['ID'].isin(sample['ID']).all()}")
print("previsões:", submission[TARGET].value_counts().sort_index().to_dict())
print("diferentes da Version04:", int((predictions != same_as_04).sum()), "de", len(predictions))

gravado em ../submissions/ML_Group_37_Version05.csv
linhas: 986, IDs únicos: 986, mesmos IDs do sample_submission: True
previsões: {0: 289, 1: 697}
diferentes da Version04: 48 de 986
